In [643]:
from fastwarc.warc import ArchiveIterator, WarcRecordType
import resiliparse.extract.html2text
from typing import Literal
import rich

WARC_FILE = "../local-shared-data/CC/example.warc.gz"
WET_FILE = "../local-shared-data/CC/example.warc.wet.gz"

In [644]:
resiliparse.extract.html2text.extract_plain_text("<html><body>Hello, world!</body></html>")

'Hello, world!'

## Code to peak through CC entries and get entries by URI (to compare WARC to WET)

In [645]:
## Retrieve a record by index or by URI 

def get_record_uri(uri, mode: Literal["wet", "warc"]):
    fname = WARC_FILE if mode == "warc" else WET_FILE
    with open(fname, "rb") as f:
        it = ArchiveIterator(f)
        val = next(it)
        while True: 
            val = next(it)
            if mode == "warc" and val.headers.get("Content-Type", "NONE") != "application/http; msgtype=response": continue
            if val.headers.get('WARC-Target-URI', "NONE") == uri: break
        return val

def get_record_ind(ind, mode: Literal["wet", "warc"]):
    fname = WARC_FILE if mode == "warc" else WET_FILE
    with open(fname, "rb") as f:
        it = ArchiveIterator(f)
        val = next(it)
        i = 0
        while i < ind:
            val = next(it)
            if mode == "warc":
                # i += 1
                if val.headers.get("Content-Type", "NONE") == "application/http; msgtype=response": i += 1
            else:
                i += 1
        return val

In [646]:
## Print some info about some retrieved records
for i in range(50, 100):
    wet_rec = get_record_ind(i, "wet")
    warc_rec = get_record_ind(i, "warc")

    print(i, warc_rec.headers.get('WARC-Target-URI', "NONE"), wet_rec.headers.get('WARC-Target-URI', "NONE"))
    print(rich.print(wet_rec.reader.read().decode()[:15]), 'len =', wet_rec.headers['Content-Length'])

50 http://adamlincoln.blogspot.com/2009/04/spain-pics.html http://afrique-infos.com/2026/02/11/the-digital-gamble-the-growth-and-influence-of-online-betting-sites/


The Digital Gam

None len = 10579
51 http://adhk.de/worglda-dayanisma-eylemi/ http://agencygarten.blogspot.com/2016/04/2016-april-topten10-model-kim-dong-kyu.html


AGENCYGARTENWOR

None len = 58
52 http://admin.koics.org/bbs/view.php?id=311&page=1 http://aguadulce-countryfair.com/tag/naki-sumo-festival/


naki sumo festi

... (78 outputs left out)

None len = 2885
92 http://archives.cshl.edu/collection/WAG/walter-gilbert-collection?q=facet,metadata.creator.en.keyword,equals,Harvard%20University%20%28subject%29&q=facet,metadata.creator.en.keyword,equals,Sidwell%20Friends%20School%20%28Washington%2C%20D.C.%29%20%28subject%29&q=facet,metadata.subject.en.keyword,equals,Film%20transparencies&q=facet,metadata.subject.en.keyword,equals,Slides%20%28Photographs%29&offset=0&limit=100&sort=metadata.AS_COMPONENT_UNIQUE_IDENTIFIER.en.keyword http://atac-inc.com/contact-us


Request A Consu

None len = 961


KeyboardInterrupt: 

In [ ]:
## Compare WARC and WET text for a fixed URI

ind = 60
uri = "http://au-urlm.com/www.arrse.co.uk"

wet_rec = get_record_uri(uri, "wet")
warc_rec = get_record_uri(uri, "warc")

# wet_rec = get_record_ind(ind, "wet")
# warc_rec = get_record_ind(ind, "warc")

rich.print(dict(warc_rec.headers))

warc_str = warc_rec.reader.read().decode()
warc_txt = resiliparse.extract.html2text.extract_plain_text(warc_str)
wet_txt = wet_rec.reader.read().decode()
print(warc_rec.headers['WARC-Target-URI'])
print(wet_rec.headers['WARC-Target-URI'])

print("WARC full:")
print(warc_str)


print("WARC text:")
print(warc_txt)

print("\n\nWET text:")
print(wet_txt)


{
    'WARC-Type': 'response',
    'WARC-Date': '2026-03-05T08:26:57Z',
    'WARC-Record-ID': '<urn:uuid:d748e2dc-bab2-47a2-8613-f6f766fe9c38>',
    'Content-Length': '16490',
    'Content-Type': 'application/http; msgtype=response',
    'WARC-Warcinfo-ID': '<urn:uuid:8037c9e0-d61b-4def-9f1b-6aac414cb992>',
    'WARC-Concurrent-To': '<urn:uuid:11f01ee6-6118-4916-9102-e3657a8ca14c>',
    'WARC-IP-Address': '54.252.111.59',
    'WARC-Target-URI': 'http://au-urlm.com/www.arrse.co.uk',
    'WARC-Protocol': 'http/1.1',
    'WARC-Payload-Digest': 'sha1:4NFYWIEZNLKBXKVB5GVORI4BSQ7BOLXO',
    'WARC-Block-Digest': 'sha1:HGVJV42RXOGYSVKHSEYURJKJSGORQCGI',
    'WARC-Identified-Payload-Type': 'application/xhtml+xml'
}

http://au-urlm.com/www.arrse.co.uk
http://au-urlm.com/www.arrse.co.uk


In [ ]:
## Test to see how many WARC files there are. There are ~60k.

with open(WARC_FILE, "rb") as f:
    it = ArchiveIterator(f)
    for i, entry in enumerate(it):
        if i % 1000 == 0:
            print(i)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000


## Make an iterator to run over WARC files and text extracted w resiliparse

In [ ]:
## Make an iterator over WARC entries, returning the entry along with extracted plain text using resiliparse
## Use as for entry, parsed_text in warc_text_iterator()
def warc_text_iterator(): 
    with open(WARC_FILE, "rb") as f:
        it = ArchiveIterator(f)
        for entry in it:
            if entry.headers.get("Content-Type", "NONE") != "application/http; msgtype=response": continue
            try: 
                warc_str = entry.reader.read().decode()
                warc_txt = resiliparse.extract.html2text.extract_plain_text(warc_str).strip()
                if len(warc_txt) == 0: continue
                yield entry, warc_txt
            except UnicodeDecodeError:
                continue

## Code to test language identification

Uses the fastText lid.176.bin  classifier

In [ ]:
import fasttext
lang_detect_model = fasttext.load_model("../local-shared-data/classifiers/lid.176.bin")

In [ ]:
def get_lang(text):
    stripped = text.replace("\n", " ").strip()
    lan, conf = lang_detect_model.predict(stripped)
    lan = lan[0]
    conf = conf[0]
    return lan, conf


def is_english(text, english_cutoff = 0.80):
    lang, conf = get_lang(text)
    return lan == "__label__en" and conf >= english_cutoff

In [ ]:
## Iterate through first 1000 WARC files. Can print text if desired. 
upto = 1000
conf_range = [0.9, 1.0]
print("CONF RANGE:", conf_range)
full_str = ""
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    
    if lan == '__label__en' and conf >= conf_range[0] and conf <= conf_range[1]: 
        print(f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}")
        print(text)
        print("\n\n--------------------\n\n")
        full_str += f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}"
        full_str += text
        full_str += "\n\n--------------------\n\n"

fname = f"../notes/conf_ran_engl_{conf_range[0]}_{conf_range[1]}"
with open(fname, 'w') as f: 
    f.write(full_str)


CONF RANGE: [0.9, 1.0]
Ind: 16, uri: http://2e-vaucresson-garches.agse.fr/2024/02/04/immediately-following-a-highly-busy-day-and-you/, length: 41095, lan: __label__en, conf: 0.9568921327590942
Aller au contenu
Site de la 2eme Vaucresson – GarchesSite de la 2eme Vaucresson - Garches
Compagnie IIème Vaucresson-Garches
  • Accueil
  • CIE 2019/2020
    • Journal
  • Contact
  • Accueil
  • CIE 2019/2020
    • Journal
  • Contact
Catégories
paras maine postimyynti morsian

Immediately following a highly busy day and you can my father being in a good car accident (he is okay, thank goodness,) it actually was

  • Auteur de l’article Par leaprouha
  • Date de l’article 4 février 2024
  • Aucun commentaire sur Immediately following a highly busy day and you can my father being in a good car accident (he is okay, thank goodness,) it actually was

Immediately following a highly busy day and you can my father being in a good car accident (he is okay, thank goodness,) it actually was

Lastly, to 

440338

(('__label__fr',), array([0.99846488]))

## Mask personal information
Write functions to mask email address, phone numbers, IP address using regexp

should replace email with |||EMAIL_ADDRESS||, etc.

In [ ]:
import re

In [ ]:
EMAIL_REGEX = r"""[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"""
PHONE_REGEX = r"""^(?:\+?\d{1,3}[-. ]?)?\(?([0-9]{3})\)?[-. ]?([0-9]{3})[-. ]?([0-9]{4})$""" 
IP_REGEX = r"""(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)""" 


COMPILED_EMAIL_REGEX = re.compile(EMAIL_REGEX)
COMPILED_PHONE_REGEX = re.compile(PHONE_REGEX)
COMPILED_IP_REGEX = re.compile(IP_REGEX)

In [ ]:
re.findall(COMPILED_IP_REGEX, "192.168.1.1 234.123.53.23")
re.findall(COMPILED_PHONE_REGEX, "+1 343 535 3421 , 646-895-2843 , 123.433.5312")

[]

In [ ]:
def mask_emails(str):
    return re.sub(COMPILED_EMAIL_REGEX, "|||EMAIL_ADDRESS|||", str)

def mask_phone(str):
    return re.sub(COMPILED_PHONE_REGEX, "|||PHONE_NUMBER|||", str)

def mask_ip(str):
    return re.sub(COMPILED_IP_REGEX, "|||IP_ADDRESS|||", str)

def mask_all(str):
    str = mask_emails(str)
    str = mask_phone(str)
    str = mask_ip(str)
    return str

In [ ]:
## Simple test. Add phone numbers.

my_str = "HELLO WORLD, my email is alexcohen@nyu.edu and another email is moo@moo.uk"
mask_emails(my_str)

'HELLO WORLD, my email is |||EMAIL_ADDRESS||| and another email is |||EMAIL_ADDRESS|||'

In [ ]:
res = list(re.finditer(COMPILED_EMAIL_REGEX, my_str))
res[0]

<re.Match object; span=(25, 42), match='alexcohen@nyu.edu'>

In [ ]:
## Iterate through first several WARC files and show regex matches
upto = 60000
buffer = 100
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    if not engl: continue

    emails = list(re.finditer(COMPILED_EMAIL_REGEX, text))
    ips = list(re.finditer(COMPILED_IP_REGEX, text))
    phones = list(re.finditer(COMPILED_PHONE_REGEX, text))

    all_matches = phones # emails + ips + phones

    if len(all_matches) > 0: 
        print(f"Ind: {i}, uri: {entry.headers.get('WARC-Target-URI', "NONE")}, length: {entry.headers['Content-Length']}, lan: {lan}, conf: {conf}")

    for match in all_matches: 
        span = match.span()
        ind1 = max(span[0] - buffer, 0)
        ind2 = min(span[1] + buffer, len(text))
        print(text[ind1:span[0]].replace('\n', ' ').strip(), "|", text[span[0]:span[1]], "|", text[span[1]:ind2].replace('\n', ' ').strip(), '\n')
    # if not (emails == [] and ips == [] and phones == []):
        
    #     print("emails:", emails)
    #     print("ips", ips)
    #     print("phones", phones)
    #     print(text)

    ## TODO add for phone numbers and IP address

Ind: 0
Ind: 100
Ind: 200
Ind: 300
Ind: 400
Ind: 500
Ind: 600
Ind: 700
Ind: 800
Ind: 900
Ind: 1000
Ind: 1100
Ind: 1200
Ind: 1300
Ind: 1400
Ind: 1500
Ind: 1600
Ind: 1700
Ind: 1800
Ind: 1900
Ind: 2000
Ind: 2100
Ind: 2200
Ind: 2300
Ind: 2400
Ind: 2500
Ind: 2600
Ind: 2700
Ind: 2800
Ind: 2900
Ind: 3000
Ind: 3100
Ind: 3200
Ind: 3300
Ind: 3400
Ind: 3500
Ind: 3600
Ind: 3700
Ind: 3800
Ind: 3900
Ind: 4000
Ind: 4100
Ind: 4200
Ind: 4300
Ind: 4400
Ind: 4500
Ind: 4600
Ind: 4700
Ind: 4800
Ind: 4900
Ind: 5000
Ind: 5100
Ind: 5200
Ind: 5300
Ind: 5400
Ind: 5500
Ind: 5600
Ind: 5700
Ind: 5800
Ind: 5900
Ind: 6000
Ind: 6100
Ind: 6200
Ind: 6300
Ind: 6400
Ind: 6500
Ind: 6600
Ind: 6700
Ind: 6800
Ind: 6900
Ind: 7000
Ind: 7100
Ind: 7200
Ind: 7300
Ind: 7400
Ind: 7500
Ind: 7600
Ind: 7700
Ind: 7800
Ind: 7900
Ind: 8000
Ind: 8100
Ind: 8200
Ind: 8300
Ind: 8400
Ind: 8500
Ind: 8600
Ind: 8700
Ind: 8800
Ind: 8900
Ind: 9000
Ind: 9100
Ind: 9200
Ind: 9300
Ind: 9400
Ind: 9500
Ind: 9600
Ind: 9700
Ind: 9800
Ind: 9900
Ind: 10000
I

## Detect NSFW content and toxic speech
```
Use /shared-data/classifiers/dolma_fastttext_hatespeech_jigsaw_model.bin for hate and toxic speech

Use /shared-data/classifiers/dolma_fasttext_nsfw_jigsaw_model.bin for NSFW content

Use fasttext library

In [ ]:
import fasttext
nsfw_model = fasttext.load_model("../local-shared-data/classifiers/dolma_fasttext_nsfw_jigsaw_model.bin")
hatespeech_model = fasttext.load_model("../local-shared-data/classifiers/dolma_fasttext_hatespeech_jigsaw_model.bin")
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [662]:
def get_is_nsfw(text, with_conf = False):
    stripped = text.replace('\n', ' ').strip()
    res = nsfw_model.predict(stripped)
    if with_conf:
        return res[0][0] == '__label__nsfw' , res[1][0]
    else:
        return res[0][0] == '__label__nsfw'

def get_is_hatespeech(text, with_conf = False):
    stripped = text.replace('\n', ' ').strip()
    res = hatespeech_model.predict(stripped)
    if with_conf:
        return res[0][0] == '__label__toxic' , res[1][0]
    else:
        return res[0][0] == '__label__toxic'

In [ ]:
nsfw_model.predict("this is a dog. it is my best friend. yay!")
nsfw_model.predict("sex porn fuck")
hatespeech_model.predict("i love dogs")
hatespeech_model.predict("i hate people")

(('__label__non-nsfw',), array([1.00000978]))

(('__label__nsfw',), array([1.00001001]))

(('__label__non-toxic',), array([1.00001001]))

(('__label__toxic',), array([1.00001001]))

In [ ]:
### Go through WARC files and show examples of NSFW and hatespeech content
## Iterate through first several WARC files and show regex matches
upto = 70000
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text)
    if not engl: continue

    is_nsfw, conf_nsfw = get_is_nsfw(text, with_conf = True)
    is_hatespeech, conf_hatespeech = get_is_hatespeech(text, with_conf=True)
    if is_nsfw: print("IS_NSFW", conf_nsfw)
    if is_hatespeech: print("IS_HATESPEECH", conf_hatespeech)
    if is_hatespeech and not is_nsfw:
        print(lan, conf)
        print(text)
        print("\n\n")

Ind: 0
Ind: 100
Ind: 200
Ind: 300
Ind: 400
Ind: 500
Ind: 600
Ind: 700
Ind: 800
Ind: 900
Ind: 1000
Ind: 1100
IS_NSFW 0.9986289739608765
IS_HATESPEECH 1.0000081062316895
Ind: 1200
Ind: 1300
Ind: 1400
Ind: 1500
Ind: 1600
Ind: 1700
Ind: 1800
Ind: 1900
Ind: 2000
Ind: 2100
Ind: 2200
Ind: 2300
Ind: 2400
Ind: 2500
Ind: 2600
Ind: 2700
Ind: 2800
Ind: 2900
Ind: 3000
Ind: 3100
Ind: 3200
Ind: 3300
Ind: 3400
Ind: 3500
Ind: 3600
Ind: 3700
Ind: 3800
Ind: 3900
Ind: 4000
Ind: 4100
Ind: 4200
Ind: 4300
Ind: 4400
Ind: 4500
Ind: 4600
Ind: 4700
Ind: 4800
Ind: 4900
Ind: 5000
Ind: 5100
Ind: 5200
Ind: 5300
Ind: 5400
Ind: 5500
Ind: 5600
Ind: 5700
Ind: 5800
Ind: 5900
Ind: 6000
Ind: 6100
Ind: 6200
Ind: 6300
Ind: 6400
Ind: 6500
Ind: 6600
Ind: 6700
Ind: 6800
Ind: 6900
Ind: 7000
Ind: 7100
Ind: 7200
Ind: 7300
Ind: 7400
Ind: 7500
Ind: 7600
Ind: 7700
Ind: 7800
Ind: 7900
Ind: 8000
IS_HATESPEECH 0.6701592803001404
__label__en 0.8946839570999146
Enjoying a walk with my beloved dog. Grateful for these simple times together.

## Implement simple Gopher quality filters 

Use nltk.word_tokenizer to count words. 

- Restrict to num words in [50, 100_000]
- Mean word length in [3, 10] chars
- Remove doc if >30% of lines end with ellipsis ...
- Remove doc is <80% of words contain at least one alphabetic character

In [ ]:
import nltk
import numpy as np

In [667]:
## Check if document passes simple gopher quality test

HAS_ALPHANUM_RE = re.compile("[a-zA-Z]")
def has_alphanum(w):
    return re.search(HAS_ALPHANUM_RE, w) != None

def check_gopher_good(text, word_ran = (50, 100_000), mean_word_len_ran = (3, 10), max_frac_ellipsis = 0.3, min_frac_alphanum = 0.8):
    words = nltk.word_tokenize(text)

    num_words_good = len(words) >= word_ran[0] and len(words) <= word_ran[1]
    
    mean_word_len = np.mean([len(w) for w in words])
    mean_word_len_good = mean_word_len >= mean_word_len_ran[0] and mean_word_len <= mean_word_len_ran[1]

    lines = text.split('\n')
    num_lines = len(lines)
    lines_with_ellipsis = len([l for l in lines if l.strip()[-3:] == "..."])
    percent_ellipsis = lines_with_ellipsis / num_lines
    ellipsis_good = percent_ellipsis <= max_frac_ellipsis

    alphanum_words = [w for w in words if has_alphanum(w)]
    percent_alphanum = len(alphanum_words)/len(words)
    alphanum_good = percent_alphanum >= min_frac_alphanum
    # print(len(words), mean_word_len, percent_ellipsis, percent_alphanum)
    return num_words_good and mean_word_len_good and ellipsis_good and alphanum_good, (len(words), mean_word_len, percent_ellipsis, percent_alphanum)

In [ ]:
## Iterate through WARC files and show ones that pass this test

gopher_pass = ""
gopher_fail = ""

upto = 2000
for i, (entry, text) in enumerate(warc_text_iterator()):
    if i % 100 == 0: print(f"Ind: {i}")
    if i > upto: break
    lan, conf = get_lang(text)
    engl = is_english(text, 0.6)
    if not engl: continue
    if get_is_nsfw(text) or get_is_hatespeech(text): continue

    pass_gopher, reason = check_gopher_good(text)

    text = mask_all(text)

    add_str = ""
    add_str += f"lan = {lan}, conf = {conf}\n"
    add_str += f"num_words = {reason[0]}, mean_word_len = {reason[1]}, percent_ellipsis = {reason[2]}, percent_alphanum = {reason[3]}\n"
    add_str += text
    add_str += "\n\n---------\n\n"
    if pass_gopher: 
        gopher_pass += add_str
    else: 
        gopher_fail += add_str


# ## TODO change to overwrite
# with open("../data_classification_tests/gopher_pass.txt", 'w') as f:
#     # f.
#     f.write(gopher_pass)

# with open("../data_classification_tests/gopher_fail.txt", 'w') as f:
#     f.write(gopher_fail)

Ind: 0
Ind: 100


## Classifier of good sites
Train a quality classifier using fasttext

Will use ```/shared-data/wiki/enwiki-20260501-extracted_urls.txt.gz``` to get a list of links from wikipedia

(Actually I don't see that file in the directory so I'm not sure where it is)

Will then subsample these to get a list of urls ```subsampled_positive_urls.txt```

Will then run 
```
wget --timeout=5 \
    -i subsampled_positive_urls.txt \
    --warc-file=subsampled_positive_urls.warc \
    -O /dev/null
```

To get those webpages into a warc file

Then will use fasttext to train a classifier, with positive examples coming from subsampled_positive_urls.warc and negatvie examples coming from random samples from common crawl

Then I will set a cutoff and keep files above that cutoff





In [669]:
## TODO

## Exact line deduplication

In [670]:
import hashlib
from pathlib import Path
 

In [ ]:
def exact_line_deduplication(inputs_paths, output_dir):
    """
    input_paths: a list of input file paths to run deduplification on 
    output_dir: a directory to write deduplified files
    For example, if the input paths are a/1.txt and a/2.txt, and the output directory is b/, this function  writes the files b/1.txt and b/2.txt. 
    """
    print(inputs_paths)
    existing_line_hashes = set([])
    for p in inputs_paths:
        with open(p, "r") as f: 
            dedup_file = ""
            for l in f.readlines():
                hashed = hashlib.sha256(l.encode('utf-8'))
                if hashed in existing_line_hashes: continue
                else: 
                    existing_line_hashes.add(hashed)
                    dedup_file += l

        new_file = Path(output_dir) / p.name
        with open(new_file, "w") as f:
            f.write(dedup_file)


In [680]:
documents_with_line_duplicates_paths = list(
            (Path("../tests/fixtures") / "documents_with_line_duplicates").glob("doc*.txt")
        )

exact_line_deduplication(documents_with_line_duplicates_paths, "new_files")

[PosixPath('../tests/fixtures/documents_with_line_duplicates/doc5.txt'), PosixPath('../tests/fixtures/documents_with_line_duplicates/doc4.txt'), PosixPath('../tests/fixtures/documents_with_line_duplicates/doc1.txt'), PosixPath('../tests/fixtures/documents_with_line_duplicates/doc3.txt'), PosixPath('../tests/fixtures/documents_with_line_duplicates/doc2.txt')]
new_files/doc5.txt
new_files/doc4.txt
new_files/doc1.txt
new_files/doc3.txt
new_files/doc2.txt


In [676]:
p1.name

'doc5.txt'